In [1]:
!pip install pytorch-tabnet2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 11.0 MB/s eta 0:00:00


In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef
)

from pytorch_tabnet import TabNetClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42

df = pd.read_csv("/kaggle/input/datasets/iqsnguyn/lancuoidibennhau/Denguediseasesdataset1003.csv_clean.csv")

FEATURES = ['Haemoglobin', 'Age', 'WBC Count', 'Differential Count', 'Platelet Count', 'PDW', 'Sex' ]
TARGET = "Final Output"

missing_cols = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Các cột sau không tồn tại trong data: {missing_cols}")

if df[TARGET].dtype == object or df[TARGET].dtype.name == 'category':
    le_target = LabelEncoder()
    y_all = le_target.fit_transform(df[TARGET])
    print(f"Target encoding: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
else:
    y_all = df[TARGET].values.astype(int)

print(f"Tổng mẫu: {len(y_all)} | Âm tính (0): {np.sum(y_all==0)} | Dương tính (1): {np.sum(y_all==1)}")

def prepare_features(df: pd.DataFrame, features: list):
    """
    Phát hiện numeric vs categorical, encode categorical bằng LabelEncoder.
    Trả về X (float32), cat_idxs, cat_dims, danh sách numeric/categorical cols.
    """
    X_df = df[features].copy()
    numeric_cols, categorical_cols = [], []

    for col in features:
        if X_df[col].dtype == object:
            categorical_cols.append(col)
        elif X_df[col].nunique() <= 15 and X_df[col].dtype in ['int32', 'int64']:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

    print(f"\nNumeric features     ({len(numeric_cols)}): {numeric_cols}")
    print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

    # Encode categorical
    for col in categorical_cols:
        le = LabelEncoder()
        X_df[col] = X_df[col].fillna("__missing__").astype(str)
        X_df[col] = le.fit_transform(X_df[col])

    cat_idxs = [features.index(c) for c in categorical_cols]
    # FIX: cat_dims phải lấy từ toàn bộ data (trước split)
    # để TabNet biết đúng số lượng category
    cat_dims = [int(X_df[c].nunique()) + 1 for c in categorical_cols]

    return X_df.values.astype(np.float32), cat_idxs, cat_dims, numeric_cols, categorical_cols


X_all, cat_idxs, cat_dims, numeric_cols, categorical_cols = prepare_features(df, FEATURES)
numeric_idxs = [FEATURES.index(c) for c in numeric_cols]

if numeric_idxs:
    imputer = SimpleImputer(strategy="median")
    X_all[:, numeric_idxs] = imputer.fit_transform(X_all[:, numeric_idxs])

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=SEED,
    stratify=y_all         
)

print(f"\n{'─'*55}")
print(f"  Train (80%): {len(y_train_full)} mẫu  "
      f"| Pos: {np.sum(y_train_full==1)} | Neg: {np.sum(y_train_full==0)}")
print(f"  Test  (20%): {len(y_test)} mẫu  "
      f"| Pos: {np.sum(y_test==1)} | Neg: {np.sum(y_test==0)}")
print(f"{'─'*55}")

def compute_class_weight(y_train):
    n_neg = np.sum(y_train == 0)
    n_pos = np.sum(y_train == 1)
    if n_pos == 0:
        raise ValueError("Fold không có mẫu dương tính!")
    return n_neg / n_pos


def threshold_by_mcc(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 199)
    best_thr, best_mcc = 0.5, -1.0
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        mcc = matthews_corrcoef(y_true, y_pred)
        if mcc > best_mcc:
            best_mcc, best_thr = mcc, thr
    return float(best_thr), float(best_mcc)


def compute_metrics(y_true, y_prob, threshold):
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "AUROC":       roc_auc_score(y_true, y_prob),
        "Accuracy":    accuracy_score(y_true, y_pred),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "PPV":         tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "NPV":         tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        "Kappa":       cohen_kappa_score(y_true, y_pred),
        "MCC":         matthews_corrcoef(y_true, y_pred)
                       if len(np.unique(y_pred)) > 1 else 0.0,
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


METRIC_COLS = ["AUROC", "Accuracy", "Sensitivity", "Specificity",
               "PPV", "NPV", "Kappa", "MCC"]


def print_metrics(metrics: dict, title: str = ""):
    if title:
        print(f"\n  {'─'*40}")
        print(f"  {title}")
        print(f"  {'─'*40}")
    for k in METRIC_COLS:
        print(f"  {k:<14} {metrics[k]:>8.4f}")
    print(f"  {'─'*24}")
    print(f"  Threshold : {metrics.get('Threshold', '—')}")
    print(f"  TP={metrics['TP']}  FP={metrics['FP']}  "
          f"TN={metrics['TN']}  FN={metrics['FN']}")

skf             = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_records    = []
fold_thresholds = []
oof_probs       = np.zeros(len(y_train_full), dtype=np.float64)

print("\n" + "=" * 65)
print(f"{'STRATIFIED 5-FOLD CV  —  trên tập TRAIN (80%)':^65}")
print("=" * 65)

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(X_train_full, y_train_full), start=1):

    print(f"\n{'─'*65}")
    print(f"  FOLD {fold_idx}/5  |  Train: {len(train_idx)}  |  Val: {len(val_idx)}")
    print(f"{'─'*65}")

    X_tr  = X_train_full[train_idx].copy()
    X_val = X_train_full[val_idx].copy()
    y_tr  = y_train_full[train_idx]
    y_val = y_train_full[val_idx]

    # Scale numeric — fit TRÊN train fold, transform cả val
    scaler = StandardScaler()
    if numeric_idxs:
        X_tr[:, numeric_idxs]  = scaler.fit_transform(X_tr[:, numeric_idxs])
        X_val[:, numeric_idxs] = scaler.transform(X_val[:, numeric_idxs])

    X_tr  = X_tr.astype(np.float32)
    X_val = X_val.astype(np.float32)

    pos_weight = compute_class_weight(y_tr)
    print(f"  pos_weight = {pos_weight:.4f}  "
          f"(neg={np.sum(y_tr==0)}, pos={np.sum(y_tr==1)})")

    clf = TabNetClassifier(
        n_d=16, n_a=16,
        n_steps=4,
        gamma=1.3,
        lambda_sparse=1e-4,
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=2,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-3, weight_decay=1e-5),
        scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params={"T_max": 50, "eta_min": 1e-5},
        verbose=0,
        seed=SEED + fold_idx,
        device_name="auto",
    )

    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric=["auc"],
        max_epochs=300,
        patience=40,
        batch_size=64,
        weights={0: 1.0, 1: float(pos_weight)},
    )

    val_prob = clf.predict_proba(X_val)[:, 1]
    oof_probs[val_idx] = val_prob

    opt_thr, opt_mcc = threshold_by_mcc(y_val, val_prob)
    fold_thresholds.append(opt_thr)

    metrics = compute_metrics(y_val, val_prob, threshold=opt_thr)
    metrics["Threshold"] = round(opt_thr, 4)
    print_metrics(metrics, title=f"Fold {fold_idx} — MCC threshold = {opt_thr:.4f}  (MCC={opt_mcc:.4f})")

    fold_records.append({
        "Fold": fold_idx,
        **{k: round(v, 4) if isinstance(v, float) else v
           for k, v in metrics.items()}
    })

mean_threshold = float(np.mean(fold_thresholds))

print(f"\n{'='*65}")
print(f"  OOF — trên tập TRAIN (80%) | mean threshold = {mean_threshold:.4f}")
print(f"  Fold thresholds: {[round(t,4) for t in fold_thresholds]}")
print(f"{'='*65}")

oof_metrics = compute_metrics(y_train_full, oof_probs, threshold=mean_threshold)
oof_metrics["Threshold"] = round(mean_threshold, 4)
print_metrics(oof_metrics, title="OOF Overall")


scaler_final = StandardScaler()
X_train_scaled = X_train_full.copy()
X_test_scaled  = X_test.copy()
if numeric_idxs:
    X_train_scaled[:, numeric_idxs] = scaler_final.fit_transform(
        X_train_full[:, numeric_idxs]
    )
    X_test_scaled[:, numeric_idxs]  = scaler_final.transform(
        X_test[:, numeric_idxs]
    )

pos_weight_final = compute_class_weight(y_train_full)
clf_final = TabNetClassifier(
    n_d=16, n_a=16,
    n_steps=4,
    gamma=1.3,
    lambda_sparse=1e-4,
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params={"T_max": 50, "eta_min": 1e-5},
    verbose=0,
    seed=SEED,
    device_name="auto",
)

print(f"\n{'─'*65}")
print(f"  Retrain trên toàn bộ TRAIN (80%) → đánh giá TEST (20%)")
print(f"{'─'*65}")

clf_final.fit(
    X_train_scaled.astype(np.float32), y_train_full,
    eval_set=[(X_test_scaled.astype(np.float32), y_test)],
    eval_metric=["auc"],
    max_epochs=300,
    patience=40,
    batch_size=64,
    weights={0: 1.0, 1: float(pos_weight_final)},
)

test_probs   = clf_final.predict_proba(X_test_scaled.astype(np.float32))[:, 1]
test_metrics = compute_metrics(y_test, test_probs, threshold=mean_threshold)
test_metrics["Threshold"] = round(mean_threshold, 4)

print_metrics(test_metrics,
              title=f"TEST SET (20%) — threshold = {mean_threshold:.4f}")


cols_order = ["Fold", "Threshold", "AUROC", "Accuracy", "Sensitivity",
              "Specificity", "PPV", "NPV", "Kappa", "MCC",
              "TP", "TN", "FP", "FN"]
fold_df = pd.DataFrame(fold_records)[cols_order]
fold_df.to_csv("fold_metrics.csv", index=False)
print(f"\n  ✓ fold_metrics.csv")

# Summary: từng fold + Mean ± Std + OOF + Test
summary_rows = []
for col in METRIC_COLS:
    vals = fold_df[col].values.astype(float)
    mean_v = np.mean(vals)
    std_v  = np.std(vals, ddof=1)         
    n      = len(vals)
    ci_lo  = mean_v - 1.96 * std_v / np.sqrt(n)
    ci_hi  = mean_v + 1.96 * std_v / np.sqrt(n)
    summary_rows.append({
        "Metric":      col,
        **{f"Fold{i}": fold_df.loc[fold_df["Fold"] == i, col].values[0]
           for i in range(1, 6)},
        "Mean":        round(mean_v, 4),
        "Std":         round(std_v, 4),
        "95CI_Lower":  round(ci_lo, 4),
        "95CI_Upper":  round(ci_hi, 4),
        "OOF":         round(float(oof_metrics[col]), 4),
        "Test":        round(float(test_metrics[col]), 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("fold_metrics_summary.csv", index=False)
print(f"  fold_metrics_summary.csv")

print(f"\n{'='*65}")
print(f"{'SUMMARY':^65}")
print(f"{'='*65}")
print(summary_df[["Metric", "Mean", "Std",
                  "95CI_Lower", "95CI_Upper",
                  "OOF", "Test"]].to_string(index=False))

print(f"\n{'='*65}")
print(f"  Split         : 80% Train / 20% Test (stratified)")
print(f"  CV            : Stratified 5-Fold trên Train")
print(f"  Threshold     : MCC-based per fold → mean = {mean_threshold:.4f}")
print(f"  Std           : sample std (ddof=1)")
print(f"{'='*65}")

Tổng mẫu: 931 | Âm tính (0): 300 | Dương tính (1): 631

Numeric features     (5): ['Haemoglobin', 'Age', 'WBC Count', 'Platelet Count', 'PDW']
Categorical features (2): ['Differential Count', 'Sex']

───────────────────────────────────────────────────────
  Train (80%): 744 mẫu  | Pos: 504 | Neg: 240
  Test  (20%): 187 mẫu  | Pos: 127 | Neg: 60
───────────────────────────────────────────────────────

          STRATIFIED 5-FOLD CV  —  trên tập TRAIN (80%)          

─────────────────────────────────────────────────────────────────
  FOLD 1/5  |  Train: 595  |  Val: 149
─────────────────────────────────────────────────────────────────
  pos_weight = 0.4764  (neg=192, pos=403)

Early stopping occurred at epoch 50 with best_epoch = 10 and best_val_0_auc = 1.0

  ────────────────────────────────────────
  Fold 1 — MCC threshold = 0.5396  (MCC=1.0000)
  ────────────────────────────────────────
  AUROC            1.0000
  Accuracy         1.0000
  Sensitivity      1.0000
  Specificity      1